# 01 — Exploratory Data Analysis

**Purpose**: Understand all raw data sources before modeling.  
**Covers**: GOV actuals, Google Trends, FRED macro, Compustat fundamentals, IBES consensus.

Run data pull scripts before executing this notebook:
```bash
python src/build_gov_table.py
python src/pull_prices.py
python src/pull_trends.py
python src/pull_fred.py
python src/pull_wrds_compustat.py
python src/pull_wrds_ibes.py
python src/build_master_df.py
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')

from src.config import (
    DASH_GOV_MASTER_PATH, MASTER_DF_PATH, GOOGLE_TRENDS_PATH,
    FRED_MACRO_PATH, COMPUSTAT_PATH, CHART_STYLE, COLORS,
)

plt.rcParams.update(CHART_STYLE)
pd.set_option('display.float_format', '{:.3f}'.format)

## 1. DASH GOV Actuals — Level and Growth

In [ ]:
gov = pd.read_csv(DASH_GOV_MASTER_PATH, parse_dates=['quarter_end_date'])
print(f"GOV master: {len(gov)} rows, {gov['quarter_end_date'].min().date()} to {gov['quarter_end_date'].max().date()}")
gov[['quarter_label', 'gov_actual_bn', 'gov_yoy_growth_pct', 'gov_surprise_pct']].tail(10)

In [ ]:
# Exhibit 1 preview: GOV trajectory
fig, ax1 = plt.subplots(figsize=(12, 5))
gov_actuals = gov.dropna(subset=['gov_actual_bn'])

ax1.bar(gov_actuals['quarter_label'], gov_actuals['gov_actual_bn'],
        color=COLORS['dash_primary'], alpha=0.6, label='GOV ($bn)')
ax1.set_ylabel('US Marketplace GOV ($bn)', color=COLORS['dash_primary'])
ax1.tick_params(axis='x', rotation=45)

ax2 = ax1.twinx()
ax2.plot(gov_actuals['quarter_label'], gov_actuals['gov_yoy_growth_pct'],
         color=COLORS['actual'], linewidth=2.5, marker='o', label='YoY Growth %')
if 'consensus_yoy_growth_pct' in gov.columns:
    ax2.plot(gov_actuals['quarter_label'], gov_actuals['consensus_yoy_growth_pct'],
             color=COLORS['consensus'], linewidth=1.5, linestyle='--', label='Consensus YoY %')
ax2.set_ylabel('YoY Growth (%)', color=COLORS['actual'])
ax2.axhline(0, color='grey', linewidth=0.5)

fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.9))
plt.title('DASH US Marketplace GOV — Levels and YoY Growth vs. Consensus')
plt.tight_layout()
plt.show()

## 2. Google Trends EDA

In [ ]:
try:
    trends = pd.read_csv(GOOGLE_TRENDS_PATH, parse_dates=['date'])
    print(f"Trends: {len(trends)} weekly rows, {trends['date'].min().date()} to {trends['date'].max().date()}")
    print(f"Columns: {list(trends.columns)}")
    trends.head(3)
except FileNotFoundError:
    print('google_trends.csv not found. Run pull_trends.py first.')

In [ ]:
# Three-way share over time
if 'three_way_doordash_share' in trends.columns:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(trends['date'], trends['three_way_doordash_share'] * 100,
            color=COLORS['dash_primary'], linewidth=1.5, label='DoorDash share %')
    ax.set_ylabel('DoorDash share of 3-way delivery search (%)')
    ax.set_title('Google Trends — DoorDash 3-Way Search Mindshare (US)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. FRED Macro EDA

In [ ]:
try:
    fred = pd.read_csv(FRED_MACRO_PATH, parse_dates=['date'])
    print(f"FRED: {len(fred)} monthly rows")
    fred.tail(6)
except FileNotFoundError:
    print('fred_macro.csv not found. Run pull_fred.py first.')

## 4. Master DataFrame Overview

In [ ]:
try:
    master = pd.read_csv(MASTER_DF_PATH, parse_dates=['quarter_end_date'])
    print(f"Master DF: {master.shape}")
    print(f"\nMissing values:")
    print(master.isna().sum()[master.isna().sum() > 0])
    master.describe().round(2)
except FileNotFoundError:
    print('master_df.csv not found. Run build_master_df.py first.')

In [ ]:
# Correlation matrix of features vs. GOV YoY growth
from src.config import OLS_BASE_FEATURES
if 'master' in dir():
    corr_cols = ['gov_yoy_growth_pct'] + [f for f in OLS_BASE_FEATURES if f in master.columns]
    corr = master[corr_cols].corr().round(2)
    print("\nCorrelation with gov_yoy_growth_pct:")
    print(corr['gov_yoy_growth_pct'].sort_values(ascending=False))